# NB02 — Feature Loading

Loads precomputed aa + kmer2 features from `enigma_stress_phenotype_ml/data/`.
Feature arrays are row-aligned with `labeled_pd.parquet` (215,051 rows, same order).
Slices train/test subsets using the protein_id indices from NB01 (row positions).

Feature combination validated in enigma_stress_phenotype_ml NB04 (mean CV AUC=0.656).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJ_ROOT = Path.cwd().parent
DATA_DIR  = PROJ_ROOT / 'data'
SRC_DATA  = PROJ_ROOT.parent / 'enigma_stress_phenotype_ml' / 'data'

# Load metadata splits from NB01
split       = json.load(open(DATA_DIR / 'split_protein_ids.json'))
train_ids   = split['train_protein_ids']  # original row positions in labeled_pd.parquet
test_ids    = split['test_protein_ids']

df_train_meta = pd.read_parquet(DATA_DIR / 'labeled_train.parquet')
df_test_meta  = pd.read_parquet(DATA_DIR / 'labeled_test.parquet')

print(f"Train: {len(df_train_meta):,} proteins ({len(train_ids)} ids)")
print(f"Test:  {len(df_test_meta):,} proteins ({len(test_ids)} ids)")

Train: 166,705 proteins (166705 ids)
Test:  48,346 proteins (48346 ids)


In [2]:
# Load precomputed features (row-aligned with labeled_pd.parquet)
aa   = pd.read_parquet(SRC_DATA / 'features_aa.parquet')
kmer = pd.read_parquet(SRC_DATA / 'features_kmer2.parquet')

print(f"AA features:    {aa.shape}")
print(f"Kmer2 features: {kmer.shape}")

aa_cols   = aa.columns.tolist()
kmer_cols = kmer.columns.tolist()
feat_cols = aa_cols + kmer_cols
print(f"Feature dimensions: {len(aa_cols)} AA + {len(kmer_cols)} kmer2 = {len(feat_cols)} total")

# Verify alignment
assert len(aa) == len(kmer) == 215051, "Feature arrays must be 215,051 rows"
print("Row-alignment check: PASS (215,051 rows in each feature file)")

AA features:    (215051, 20)
Kmer2 features: (215051, 400)
Feature dimensions: 20 AA + 400 kmer2 = 420 total
Row-alignment check: PASS (215,051 rows in each feature file)


In [3]:
# Cross-file alignment verification (beyond row count)
# Feature files have no protein_id column — alignment is implicit by row position.
# Verify that the row positions saved by NB01 still map to the same organisms in labeled_pd.parquet.
labeled_ref = pd.read_parquet(SRC_DATA / 'labeled_pd.parquet', columns=['organism'])
assert len(labeled_ref) == 215051, f"labeled_pd row count changed: {len(labeled_ref)} (expected 215,051)"
# Spot-check: organism at row positions train_ids[:10] must match NB01's saved labeled_train.parquet
ref_orgs = labeled_ref['organism'].iloc[train_ids[:10]].tolist()
saved_orgs = df_train_meta['organism'].iloc[:10].tolist()
assert ref_orgs == saved_orgs, (
    "Organism mismatch at train positions 0-9 — feature files may not match this labeled_pd.parquet.\n"
    f"Expected: {saved_orgs}\nGot:      {ref_orgs}"
)
print(f"labeled_pd row count: {len(labeled_ref):,} rows (matches feature files)")
print(f"Organism spot-check: position {train_ids[0]} → '{ref_orgs[0]}' (matches NB01 saved split)")
print("Cross-file alignment check: PASS (positional mapping consistent across labeled_pd and saved split)")

labeled_pd row count: 215,051 rows (matches feature files)
Organism spot-check: position 0 → 'acidovorax_3H11' (matches NB01 saved split)
Cross-file alignment check: PASS (positional mapping consistent across labeled_pd and saved split)


In [4]:
# Positional slicing — train_ids / test_ids are original row positions in labeled_pd
aa_train   = aa.iloc[train_ids].reset_index(drop=True)
kmer_train = kmer.iloc[train_ids].reset_index(drop=True)
aa_test    = aa.iloc[test_ids].reset_index(drop=True)
kmer_test  = kmer.iloc[test_ids].reset_index(drop=True)

X_train_raw = pd.concat([aa_train, kmer_train], axis=1).values.astype(np.float32)
X_test_raw  = pd.concat([aa_test,  kmer_test],  axis=1).values.astype(np.float32)

print(f"X_train_raw: {X_train_raw.shape}")
print(f"X_test_raw:  {X_test_raw.shape}")

# Verify alignment with label splits
assert X_train_raw.shape[0] == len(df_train_meta), "Train feature/label size mismatch"
assert X_test_raw.shape[0]  == len(df_test_meta),  "Test feature/label size mismatch"
print("Shape alignment check: PASS")

X_train_raw: (166705, 420)
X_test_raw:  (48346, 420)
Shape alignment check: PASS


In [5]:
# Standardize (fit on train only — no test leakage)
train_mean = X_train_raw.mean(axis=0)
train_std  = X_train_raw.std(axis=0)
train_std[train_std == 0] = 1.0  # constant columns

X_train = (X_train_raw - train_mean) / train_std
X_test  = (X_test_raw  - train_mean) / train_std

y_train      = df_train_meta['essential_union'].values
y_test       = df_test_meta['essential_union'].values
orgs_train   = df_train_meta['organism'].values
genera_train = df_train_meta['genus'].values
orgs_test    = df_test_meta['organism'].values

print(f"X_train: {X_train.shape}  y_train: {y_train.shape} (pos={y_train.sum()}, rate={100*y_train.mean():.2f}%)")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}  (pos={y_test.sum()}, rate={100*y_test.mean():.2f}%)")
print(f"Training genera: {len(np.unique(genera_train))}  organisms: {len(np.unique(orgs_train))}")

X_train: (166705, 420)  y_train: (166705,) (pos=22406, rate=13.44%)
X_test:  (48346, 420)   y_test:  (48346,)  (pos=4655, rate=9.63%)
Training genera: 32  organisms: 48


In [6]:
# Save arrays for NB03
np.save(DATA_DIR / 'X_train.npy', X_train)
np.save(DATA_DIR / 'X_test.npy',  X_test)
np.save(DATA_DIR / 'y_train.npy', y_train)
np.save(DATA_DIR / 'y_test.npy',  y_test)
np.save(DATA_DIR / 'orgs_train.npy',   orgs_train.astype(str))
np.save(DATA_DIR / 'genera_train.npy', genera_train.astype(str))
np.save(DATA_DIR / 'orgs_test.npy',    orgs_test.astype(str))

json.dump(feat_cols,      open(DATA_DIR / 'feature_columns.json', 'w'))
json.dump({'mean': train_mean.tolist(), 'std': train_std.tolist()},
          open(DATA_DIR / 'scaler_params.json', 'w'))

print(f"Saved all arrays to {DATA_DIR}")

Saved all arrays to /home/hmacgregor/BERIL-research-observatory/projects/enigma_general_essentiality/data
